# Game Simulator

In [8]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.sheets import load_inputs, get_inputs_dir
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel
from common_lib.tables import monthly_table, comparison_table, export_all_tables, comparison_table
import datetime as dt
#print('Inputs dir:', get_inputs_dir())

In [9]:
refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle

In [10]:
# show
bqc = BigQueryConnector()
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}
#cost = bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)

In [11]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}
actuals = pd.DataFrame()
if refresh_data:
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=params)
    pd.to_pickle(actuals, './data/actuals.pkl')
else:
    actuals = pd.read_pickle('./data/actuals.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()

In [12]:
actuals

,dt,platform,dau,new_installs,iap_revenue,iap_net_revenue,ad_revenue,ad_net_revenue
0,2021-06-21,android,5,5,41.465483,29.025838,NaN,NaN
1,2021-06-21,ios,4,4,NaN,NaN,NaN,NaN
2,2021-06-22,android,129,127,NaN,NaN,NaN,NaN
3,2021-06-22,ios,4,3,NaN,NaN,NaN,NaN
4,2021-06-23,android,664,610,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
3635,2026-06-12,ios,45565,708,12709.964927,9575.505346,4752.006393,4039.205434
3636,2026-06-13,android,31017,413,10500.814162,7790.607414,3481.967759,2959.672595
3637,2026-06-13,ios,44743,728,15728.647441,11800.046854,5279.720233,4487.762198
3638,2026-06-14,android,31774,405,11140.859447,8210.333483,3807.488454,3236.365186


In [5]:
# show

#bqc = BigQueryConnector()
#cost = bqc.print_cost_estimate('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
#cost = bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

In [13]:
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}

if refresh_data:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

In [14]:
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}

if refresh_data:
    marketing  = bqc.get('./sql/marketing.sql',  is_path=True, query_parameters=params)
    marketing.to_pickle('./data/marketing.pkl')
else:
    marketing  = pd.read_pickle('./data/marketing.pkl')


In [15]:
# show
sheet_inputs = load_inputs()

#for name, df in sheet_inputs.items():
    #print(f'\n--- {name} ---')
    #print(df.to_string(index=False))

In [16]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios(), last_actuals_date=actuals['dt'].dt.date.max())
prefill_panel(panel, actuals, anchor_dau, sheet_inputs)
setup_callbacks(panel, engine, actuals,
                live_retention=live_retention, live_conversion=live_conversion,
                installs=actuals)
panel.display()

<IPython.core.display.Javascript object>

In [ ]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
#print('Available results:', list_results())

In [ ]:
#plot('test3', chart='dau')

In [ ]:
#plot('test3', chart='revenue')

In [ ]:


results = comparison_table(actuals=actuals)

#round results for better display on pandas
results['Cumul Margin 2027-03'] = results['Cumul Margin 2027-03'].round(0)
results['Cumul Margin 2027-12'] = results['Cumul Margin 2027-12'].round(0)
results['Cumul Margin 2029-12'] = results['Cumul Margin 2029-12'].round(0)


In [ ]:
#results.sort_values('Cumul Margin 2027-12', ascending=False)

In [ ]:
#export_all_tables(actuals=actuals)